# PHẦN 17: IMPORT THƯ VIỆN PHỤC VỤ MACHINE LEARNING

## 1. Mục tiêu
Thiết lập môi trường lập trình và tích hợp các công cụ hỗ trợ huấn luyện mô hình dự báo. Các thư viện được sử dụng thực tế trong bài bao gồm:

* **Quản lý & Xử lý dữ liệu**: `pandas` (quản lý DataFrame), `numpy` (tính toán số học).
* **Tiền xử lý dữ liệu (Scikit-learn)**:
    - `SimpleImputer`: Xử lý giá trị khuyết thiếu.
    - `RobustScaler`: Chuẩn hóa dữ liệu chống nhiễu (Outliers).
    - `ColumnTransformer` & `Pipeline`: Xây dựng quy trình xử lý dữ liệu tự động.
* **Chia tập dữ liệu & Tối ưu hóa**:
    - `train_test_split`: Phân tách dữ liệu Train/Test.
    - `RandomizedSearchCV`: Tìm kiếm siêu tham số tối ưu và thực hiện Cross-Validation tích hợp.
    
* **Thuật toán Machine Learning**:
    - `Ridge`: Hồi quy tuyến tính có hiệu chỉnh L2.
    - `RandomForestRegressor`: Thuật toán rừng ngẫu nhiên.
    - `XGBRegressor`: Thuật toán Boosting hiệu suất cao.
* **Đánh giá & Lưu trữ**:
    - `mean_absolute_error`, `mean_squared_error`, `r2_score`: Các chỉ số đo lường sai số.
    - `joblib`: Xuất và lưu trữ mô hình sau huấn luyện.
* **Trực quan hóa**: `matplotlib.pyplot` và `seaborn`.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
import warnings
warnings.filterwarnings('ignore')

# PHẦN 18: ĐỌC DỮ LIỆU ĐÃ TIỀN XỬ LÝ

## 2. Tải Dataset sau quá trình EDA

Dataset đã qua các bước xử lý quan trọng từ notebook trước sẽ được tải lại để phục vụ huấn luyện:
* **Cleaning**: Làm sạch dữ liệu nhiễu.
* **Feature Engineering**: Tạo các đặc trưng mới.
* **Encoding**: Mã hóa các biến phân loại.
* **Log Transformation**: Biến đổi logarit để xử lý dữ liệu lệch.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Khai pha DL TLU/Cuối kỳ/Data btl datamining/clean_housepredict_data.csv')
df.head()

In [ ]:
df.shape

# Business Understanding

## Bối cảnh bài toán

Thị trường bất động sản tại Hà Nội và TP.HCM là hai trong những thị trường sôi động và phức tạp nhất Việt Nam. Giá nhà phụ thuộc vào hàng chục yếu tố đan xen nhau: vị trí địa lý, diện tích, cấu trúc công trình, tiện ích nội thất, loại hình bất động sản và xu hướng thị trường từng thời điểm.

Trong thực tế, việc định giá nhà thường dựa vào kinh nghiệm chủ quan của môi giới hoặc so sánh thủ công với các tin rao lân cận — phương pháp này tốn thời gian và dễ bị sai lệch do lợi ích cá nhân.

## Mục tiêu dự án

Dự án nhằm xây dựng một **mô hình dự đoán giá nhà tự động** dựa trên các đặc trưng khách quan của bất động sản, hướng đến hai mục tiêu cụ thể:

1. **Dự đoán giá**: Ước tính mức giá hợp lý (tính bằng tỷ đồng) cho một căn nhà dựa trên các thông số đầu vào như diện tích, số tầng, vị trí, tiện ích…
2. **Phân tích yếu tố**: Xác định những đặc trưng nào ảnh hưởng nhiều nhất đến giá nhà, từ đó đưa ra khuyến nghị thực tiễn cho người mua, người bán và các nền tảng giao dịch bất động sản.

## Phạm vi

- **Địa bàn**: Hà Nội và TP.HCM
- **Phân khúc**: Nhà ở dân dụng (nhà ống, nhà phố) giá dưới 15 tỷ đồng
- **Nguồn dữ liệu**: Thu thập qua web scraping từ các nền tảng giao dịch bất động sản công khai
- **Loại bài toán**: Hồi quy (Regression) — biến mục tiêu là `price` liên tục



### 2.1. Chi tiết các đặc trưng (Features Dictionary)

Dựa trên cấu trúc dữ liệu từ `df.info()`, dưới đây là ý nghĩa của các biến trong bộ dữ liệu:

| Tên đặc trưng | Ý nghĩa / Vai trò |
| :--- | :--- |
| `price` | **Biến mục tiêu (Target)**: Giá nhà đã qua biến đổi Logarit. |
| `area` | Diện tích của căn nhà. |
| `floor_number` | Tổng số tầng của căn nhà. |
| `bedroom_number` | Số lượng phòng ngủ. |
| `is_dinning_room` | Biến nhị phân: Có phòng ăn hay không (1: Có, 0: Không). |
| `is_kitchen` | Biến nhị phân: Có nhà bếp hay không. |
| `is_terrace` | Biến nhị phân: Có sân thượng hay không. |
| `is_car_pack` | Biến nhị phân: Có chỗ đỗ xe ô tô hay không. |
| `type` | Loại hình bất động sản (Nhà mặt phố hoặc nhà hẻm). |
| `street_in_front_of_house` | Độ rộng của mặt đường trước nhà. |
| `width` | Độ rộng mặt tiền của căn nhà. |
| `city` | Thành phố (Hà Nội hoặc TP.HCM). |
| `district` | Quận/Huyện nơi bất động sản tọa lạc. |

## 3. Kiểm tra dữ liệu trùng lặp
Chúng ta cần xác định số lượng bản ghi bị lặp để đảm bảo tính khách quan cho mô hình.

In [ ]:
df.duplicated().sum()

In [ ]:
df.duplicated().sum()/df.shape[0]*100

## 4. Xử lý trùng lặp và Ngăn chặn Data Leakage

Qua kiểm tra, dự án phát hiện **8.656** bản ghi bị trùng lặp hoàn toàn. Dự án quyết định loại bỏ 100% các bản ghi dư thừa, chỉ giữ lại bản ghi gốc đầu tiên (`keep='first'`).

**Quyết định này dựa trên 2 yếu tố trọng yếu:**

1. **Tri thức ngành (Domain Knowledge)**: Dữ liệu thu thập qua **Web Scraping** thường xuyên gặp hiện tượng "Spam tin đăng" (một căn nhà được đăng nhiều lần).
2. **Ngăn chặn Rò rỉ Dữ liệu (Data Leakage)**: Việc giữ lại các bản sao sẽ khiến các bản ghi giống hệt nhau xuất hiện ở cả tập **Train** và **Test**. Điều này dẫn đến hiện tượng **Overfitting** nội tại — mô hình có độ chính xác ảo trên tập Test nhưng thực tế hoạt động kém hiệu quả.

In [ ]:
# Tiến hành xóa và chỉ giữ lại dòng đầu tiên xuất hiện
df.drop_duplicates(keep='first', inplace=True)

In [ ]:
df.info()

# PHẦN 19: XỬ LÝ DỮ LIỆU KHUYẾT THIẾU (MISSING DATA)

## 1. Chiến lược Imputation
Hướng đi phổ biến là sử dụng phương pháp **Imputation** (điền giá trị thay thế dựa trên `Mean`, `Median` hoặc `Mode`). Tuy nhiên, cần cân nhắc kỹ cho từng đặc trưng cụ thể.

## 2. Rủi ro khi Impute đặc trưng `width` (Mặt tiền)

* **Biến dạng phân bố dữ liệu**: Với hơn 10.000 dòng khuyết (1/4 dữ liệu), việc điền `Mean` hoặc `Median` sẽ tạo ra một lượng lớn căn nhà có kích thước ảo giống hệt nhau, phá hỏng quy luật tự nhiên của thị trường.
* **Làm sai lệch giá nhà**: Kích thước mặt tiền là yếu tố cốt lõi định giá. Việc nội suy ảo sẽ khiến mô hình học sai quy luật, dẫn đến dự báo sai lệch hoàn toàn.

### Phân tích Cơ chế khuyết thiếu cho đặc trưng Width

Trước khi đưa ra quyết định xử lý hơn 10.000 giá trị khuyết (NaN) của đặc trưng `width` (Mặt tiền), dự án đã tiến hành phân tích độ tương quan giữa việc khuyết thiếu dữ liệu này với loại hình bất động sản (`type`).



In [ ]:

# 1. Đếm số lượng nhà bị thiếu 'width' theo từng loại hình (type)
missing_width_counts = df[df['width'].isna()]['type'].value_counts()

# 2. Đếm tổng số nhà của mỗi loại hình
total_type_counts = df['type'].value_counts()

# 3. Tính tỷ lệ % thiếu
missing_ratio = (missing_width_counts / total_type_counts * 100).round(2)

missing_summary = pd.DataFrame({
    'Tổng số lượng': total_type_counts,
    'Số lượng khuyết Width': missing_width_counts,
    'Tỷ lệ khuyết (%)': missing_ratio
})

print("Bảng thống kê tỷ lệ khuyết thiếu 'width' theo Loại hình nhà:")
print(missing_summary)

### 2.1. Kết quả thống kê khuyết thiếu
* **Nhóm `type 0` (Nhà trong hẻl)**: Tỷ lệ khuyết **27.46%**.
* **Nhóm `type 1` (Nhà mặt tiền)**: Tỷ lệ khuyết **26.95%**.

## 3. Thực thi Xử lý Dữ liệu Khuyết thiếu (Hybrid Imputation)

Dự án áp dụng quy trình 2 bước:
1. **Loại bỏ**: Xóa các quan trắc bị khuyết `width` và `street_in_front_of_house` để tránh gây nhiễu không gian đặc trưng.
2. **Impute**: Đối với các đặc trưng khuyết ít như `area`, `floor_number`, `bedroom_number`, sử dụng `SimpleImputer` với chiến lược `median` trong Pipeline.

In [ ]:
# Xóa các dòng bị khuyết 'width' và 'street_in_front_of_house'
df.dropna(subset=['width', 'street_in_front_of_house'], inplace=True)

In [ ]:
df.shape

# PHẦN 20: LẤY MẪU PHÂN TẦNG (STRATIFIED SAMPLING)

## 1. Mục tiêu lấy mẫu
Để tối ưu thời gian huấn luyện mà vẫn giữ được tính đại diện, nhóm sử dụng **Stratified Sampling** dựa trên các khoảng giá nhà (`price_bin`).

## 2. Phương pháp thực hiện
* Sử dụng `pd.qcut` để chia biến `price` thành 5 nhóm bằng nhau.
* Lấy mẫu ngẫu nhiên **5.000 dòng** đảm bảo tỷ lệ các nhóm giá không đổi.
* Giúp bảo toàn đặc điểm phân phối và hạn chế mất cân bằng dữ liệu.

In [ ]:
# Tạo các nhóm giá
df['price_bin'] = pd.qcut(
    df['price'],
    q=5,
    labels=False
)

# Stratified sampling lấy 5000 dòng
df_sample, _ = train_test_split(
    df,
    train_size=5000,
    stratify=df['price_bin'],
    random_state=42
)

# Xóa cột phụ
# df_sample = df_sample.drop(columns='price_bin')

print(df_sample.shape)

### 2.1. So sánh phân phối giá nhà

Biểu đồ Histogram cho thấy phân phối của biến `price` sau khi lấy mẫu gần như không thay đổi so với tập dữ liệu gốc. Điều này chứng minh phương pháp **Stratified Sampling** đã đạt được hiệu quả bảo toàn đặc tính dữ liệu.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# So sánh phân phối price trước và sau stratified sampling

plt.figure(figsize=(10, 6))

sns.histplot(
    df['price'],
    bins=40,
    stat='density',
    kde=True,
    alpha=0.5,
    label='Dataset gốc'
)

sns.histplot(
    df_sample['price'],
    bins=40,
    stat='density',
    kde=True,
    alpha=0.5,
    label='Dataset sau sampling'
)

plt.xlabel('Price')
plt.ylabel('Density')
plt.title('So sánh phân phối Price trước và sau Stratified Sampling')

plt.legend()

plt.show()

In [ ]:
df_sample.isna().sum()

# PHẦN 21: HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH

## 1. Chia tập Train / Test
Tập dữ liệu được chia theo tỷ lệ **80/20** và vẫn áp dụng **Stratified Split** dựa trên `price_bin` để đảm bảo tập Test phản ánh đúng cấu trúc của tập Train.

In [ ]:
x = df_sample.drop(columns=['price', 'price_bin'])
y = df_sample['price']

# ======================
# Train test split
# ======================

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    stratify=df_sample['price_bin'],
    random_state=42
)


## 2. Chuẩn hóa dữ liệu với RobustScaler

Nhóm lựa chọn `RobustScaler` thay vì `StandardScaler` vì:
* **Chống Outlier**: Sử dụng trung vị (`Median`) và khoảng tứ phân vị (`IQR`) nên ít bị ảnh hưởng bởi các giá trị cực đoan.
* **Dữ liệu bất động sản**: Phù hợp với đặc thù dữ liệu giá nhà vốn thường có độ lệch lớn.

In [ ]:
num_cols = [
    'floor_number',
    'bedroom_number'
]

In [ ]:
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_cols)
], remainder='passthrough')

# ======================
# Fit transform
# ======================

x_train = preprocessor.fit_transform(x_train)

x_test = preprocessor.transform(x_test)

print(x_train.shape)
print(x_test.shape)

## 3. Lựa chọn mô hình (Model Selection)

### A. Các mô hình không sử dụng
* **KNN Regressor**: Không hiệu quả trên dữ liệu chứa nhiều biến nhị phân và chi phí tính toán cao.
* **SVR**: Thời gian huấn luyện và tinh chỉnh quá lớn cho tập dữ liệu này.

### B. Các mô hình được lựa chọn
1. **Ridge Regression**: Mô hình Baseline để đánh giá tính tuyến tính, cực kỳ dễ giải thích.
2. **Random Forest Regressor**: Học tốt các quan hệ phi tuyến và cung cấp `Feature Importance`.
3. **XGBoost Regressor**: Đại diện cho sức mạnh dự báo tối đa thông qua cơ chế **Boosting** sửa lỗi liên tục.

### 3.1. Ridge Regression với `RandomizedSearchCV`

**Ridge Regression** bổ sung kỹ thuật **Regularization L2** vào mô hình hồi quy tuyến tính truyền thống, giúp phạt các hệ số (`coefficients`) quá lớn.

* **Ưu điểm**: Khi các đặc trưng có sự tương quan cao (ví dụ: `area` và `bedroom_number`), mô hình **Ridge** sẽ hoạt động ổn định hơn so với `LinearRegression` thuần túy.
* **Tối ưu hóa**: Tham số `alpha` điều chỉnh mức độ phạt — chúng ta thực hiện tìm kiếm ngẫu nhiên trên dải rộng từ `0.01` đến `100` để xác định điểm cân bằng giữa **Bias** và **Variance**.

In [ ]:
# Ridge Pipeline

ridge_pipeline = Pipeline([
    ('model', Ridge())
])


# Hyperparameter

ridge_params = {
    'model__alpha': [0.01, 0.1, 1, 5, 10, 50, 100]
}

# Randomized Search CV

ridge_search = RandomizedSearchCV(
    estimator=ridge_pipeline,
    param_distributions=ridge_params,
    n_iter=7,
    cv=5,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1
)

# Train model

ridge_search.fit(x_train, y_train)

# Best model

best_ridge_model = ridge_search.best_estimator_

# Predict

ridge_pred = best_ridge_model.predict(x_test)

# Evaluation

ridge_mae = mean_absolute_error(y_test, ridge_pred)

# Calculate RMSE
ridge_mse = mean_squared_error(y_test, ridge_pred)
ridge_rmse = ridge_mse ** 0.5

ridge_r2 = r2_score(y_test, ridge_pred)

# Result

print("========== Ridge Regression ==========")

print(f"Best Alpha : {ridge_search.best_params_['model__alpha']}")
print(f"MAE        : {ridge_mae:.4f}")
print(f"RMSE       : {ridge_rmse:.4f}")
print(f"R2 Score   : {ridge_r2:.4f}")
print(f"Best CV    : {-ridge_search.best_score_:.4f}")

### 3.2. Kết quả huấn luyện Ridge Regression

| Chỉ số đánh giá | Giá trị | Ý nghĩa |
| :--- | :---: | :--- |
| **MAE** | 0.2285 | Sai số tuyệt đối trung bình trên tập kiểm thử |
| **RMSE** | 0.3203 | Sai số căn bậc hai trung bình (phạt nặng các lỗi lớn) |
| **R² Score** | 0.4731 | Mô hình giải thích được ~47.31% biến động dữ liệu |
| **Best CV (MAE)** | 0.2177 | Sai số trung bình trong quá trình **Cross-Validation** |

#### ⚙️ Siêu tham số tối ưu (Best Hyperparameters)
* **`alpha`**: `0.01` (Mức độ phạt **L2** rất nhỏ, mô hình có xu hướng tiệm cận về hồi quy tuyến tính gốc).

### 3.3. Random Forest Regressor

**Cơ chế hoạt động:**

**Random Forest** là thuật toán **Ensemble Learning** xây dựng hàng trăm cây quyết định (**Decision Trees**) chạy song song.

* **Bootstrap Sampling**: Mỗi cây được huấn luyện trên một tập dữ liệu con được lấy mẫu ngẫu nhiên có hoàn lại.
* **Feature Subsampling**: Tại mỗi nút phân chia, mô hình chỉ xem xét một tập hợp con ngẫu nhiên các đặc trưng.
* **Kết quả**: Dự đoán cuối cùng là trung bình cộng của tất cả các cây con, giúp triệt tiêu phương sai và giảm thiểu rủi ro **Overfitting**.

**Không gian siêu tham số (Hyperparameter Grid):**

* `n_estimators`: Số lượng cây quyết định trong rừng.
* `max_depth`: Giới hạn độ sâu tối đa của cây.
* `min_samples_split`: Số lượng mẫu tối thiểu cần thiết để tiếp tục chẻ nhánh.
* `min_samples_leaf`: Số lượng mẫu tối thiểu bắt buộc phải có ở một nút lá.
* `max_features`: Số lượng đặc trưng tối đa được trích xuất ngẫu nhiên để xem xét tại mỗi lần phân chia.

In [ ]:

# Random Forest Pipeline

rf_pipeline = Pipeline([
    ('model', RandomForestRegressor(random_state=42))
])

# Hyperparameter

rf_params = {
    'model__n_estimators': [100, 250, 500, 750],
    'model__max_depth': [5, 8, 10, 15, None],
    'model__min_samples_split': [2, 4, 6, 10, 20],
    'model__min_samples_leaf': [1, 3, 5, 7],
    'model__max_features': ['sqrt', 'log2', 3, 5, 7]
}

# Randomized Search CV

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_params,
    n_iter=30,
    cv=5,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Train

rf_search.fit(x_train, y_train)


# Best model

best_rf_model = rf_search.best_estimator_

# Predict

rf_pred = best_rf_model.predict(x_test)

# Evaluation

rf_mae = mean_absolute_error(y_test, rf_pred)

# Calculate RMSE
rf_mse = mean_squared_error(y_test, rf_pred)
rf_rmse = rf_mse ** 0.5

rf_r2 = r2_score(y_test, rf_pred)

# Result

print("========== Random Forest ==========")

print(f"Best Params : {rf_search.best_params_}")
print(f"MAE         : {rf_mae:.4f}")
print(f"RMSE        : {rf_rmse:.4f}")
print(f"R2 Score    : {rf_r2:.4f}")
print(f"Best CV     : {-rf_search.best_score_:.4f}")

### 3.4. Kết quả huấn luyện Random Forest

| Chỉ số đánh giá | Giá trị | Ý nghĩa |
| :--- | :---: | :--- |
| **MAE** | 0.2003 | Sai số tuyệt đối trung bình trên tập kiểm thử |
| **RMSE** | 0.2788 | Sai số căn bậc hai trung bình |
| **R² Score** | 0.6006 | Mô hình giải thích được ~60.06% biến động dữ liệu |
| **Best CV (MAE)** | 0.1965 | Sai số trung bình trong quá trình **Cross-Validation** |

#### ⚙️ Bộ siêu tham số tối ưu (Best Hyperparameters)
* **`n_estimators`**: `100`
* **`max_depth`**: `None` (Cây phát triển tự do đến khi đạt điều kiện dừng)
* **`max_features`**: `5` (Số tính năng tối đa được xem xét tại mỗi nút)
* **`min_samples_split`**: `10`
* **`min_samples_leaf`**: `1`

### 3.5. XGBoost Regressor

**XGBoost** (Extreme Gradient Boosting) được kỳ vọng mang lại sai số thấp nhất cho dự án nhờ cơ chế tối ưu hóa gradient mạnh mẽ.

**Chiến lược tinh chỉnh siêu tham số:**

* `n_estimators` & `learning_rate`: Cặp tham số kiểm soát tốc độ hội tụ và độ chính xác của quá trình học.
* `max_depth`: Giới hạn độ sâu để tránh hiện tượng "học vẹt".
* `min_child_weight`: Giúp ngăn chặn việc chia nhỏ các nút lá quá mức, tương tự `min_samples_leaf`.
* `subsample` & `colsample_bytree`: Các kỹ thuật lấy mẫu ngẫu nhiên theo dòng và cột để tăng cường tính **Generalization** (tổng quát hóa) và chống **Overfitting**.

In [ ]:
# XGBoost Pipeline

xgb_pipeline = Pipeline([
    ('model', XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        verbosity=0
    ))
])

# Hyperparameter

xgb_params = {
    'model__n_estimators': [100, 250, 500, 750, 1000],
    'model__max_depth': [2, 3, 4, 5, 6, 7],
    'model__learning_rate': [0.15, 0.1, 0.05, 0.01],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'model__min_child_weight': [1, 3, 5, 7]
}


# Randomized Search CV

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_params,
    n_iter=30,
    cv=5,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Train

xgb_search.fit(x_train, y_train)

# Best model

best_xgb_model = xgb_search.best_estimator_

# Predict


xgb_pred = best_xgb_model.predict(x_test)

# Evaluation

xgb_mae = mean_absolute_error(y_test, xgb_pred)

#  Calculate RMSE m
xgb_mse = mean_squared_error(y_test, xgb_pred)
xgb_rmse = xgb_mse ** 0.5

xgb_r2 = r2_score(y_test, xgb_pred)

# Result

print("========== XGBoost ==========")

print(f"Best Params : {xgb_search.best_params_}")
print(f"MAE         : {xgb_mae:.4f}")
print(f"RMSE        : {xgb_rmse:.4f}")
print(f"R2 Score    : {xgb_r2:.4f}")
print(f"Best CV     : {-xgb_search.best_score_:.4f}")

### 3.6. Kết quả huấn luyện XGBoost

| Chỉ số đánh giá | Giá trị | Ý nghĩa |
| :--- | :---: | :--- |
| **MAE** | 0.1984 | Sai số tuyệt đối trung bình trên tập kiểm thử |
| **RMSE** | 0.2761 | Sai số căn bậc hai trung bình |
| **R² Score** | 0.6082 | Mô hình giải thích được ~60.82% biến động dữ liệu |
| **Best CV (MAE)** | 0.1920 | Sai số trung bình trong quá trình **Cross-Validation** |

#### ⚙️ Bộ siêu tham số tối ưu (Best Hyperparameters)
* **`n_estimators`**: `750`
* **`max_depth`**: `7` (Độ sâu tối ưu để bắt các quan hệ phi tuyến phức tạp)
* **`learning_rate`**: `0.01` (Tốc độ học nhỏ giúp hội tụ mịn và chuẩn xác)
* **`subsample`**: `0.7` | **`colsample_bytree`**: `0.8` | **`min_child_weight`**: `1`

# PHẦN 22: TỔNG HỢP VÀ LỰA CHỌN MÔ HÌNH (CHAMPION MODEL)

## 1. Phương pháp đánh giá

Sau khi tối ưu hóa bằng `RandomizedSearchCV`, chúng ta thực hiện so sánh hiệu suất dựa trên hệ đo lường đa chiều:

1. **Độ đo Tối ưu hóa**: `MAE` (Mean Absolute Error) được chọn làm thước đo chính vì tính ổn định và dễ giải thích.
2. **Độ đo So sánh**: Bổ sung `RMSE` và `R² Score` để có cái nhìn toàn diện.
3. **Đánh giá Quá khớp**: Tính toán **Generalization Gap** (`CV MAE` - `Test MAE`). Mô hình lý tưởng cần có hiệu suất cao và khoảng cách này tiến sát về 0.

## 2. Bảng so sánh hiệu năng tổng thể

Chúng ta tổng hợp kết quả của cả 3 mô hình theo các tiêu chí trọng yếu:

* **R² (Test set)**: Tỷ lệ phương sai giá nhà được mô hình giải thích.
* **RMSE (Test set)**: Sai số trên thang log, nhạy cảm với các sai lệch lớn.
* **MAE (Test set)**: Sai số tuyệt đối trung bình.
* **CV MAE**: Đánh giá độ ổn định của mô hình trên các tập dữ liệu con khác nhau.

In [ ]:
# CV MAE
ridge_cv_mae = -ridge_search.best_score_
rf_cv_mae = -rf_search.best_score_
xgb_cv_mae = -xgb_search.best_score_

# Result Table
results = pd.DataFrame({
    'Model': ['Ridge', 'Random Forest', 'XGBoost'],
    'Test R2': [ridge_r2, rf_r2, xgb_r2],
    'Test RMSE': [ridge_rmse, rf_rmse, xgb_rmse],
    'Test MAE': [ridge_mae, rf_mae, xgb_mae],
    'CV MAE': [ridge_cv_mae, rf_cv_mae, xgb_cv_mae]
})

# Generalization Gap
results['Generalization Gap'] = results['CV MAE'] - results['Test MAE']

# Sort
results = results.sort_values(by='Test R2', ascending=False).round(4).reset_index(drop=True)
print(results.to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ============================================
# Create Figure
# ============================================

fig, (ax1, ax2, ax3) = plt.subplots(
    1, 3,
    figsize=(16, 5)
)

model_names = results['Model']

colors = [
    '#2196F3',
    '#4CAF50',
    '#FF9800'
]

# ============================================
# 1. R2 Score
# ============================================

bars_r2 = ax1.bar(
    model_names,
    results['Test R2'],
    color=colors
)

ax1.set_title('R2 Score Comparison')
ax1.set_ylabel('R2 Score')

for bar in bars_r2:

    ax1.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.003,
        f'{bar.get_height():.4f}',
        ha='center',
        fontsize=8
    )

# ============================================
# 2. RMSE
# ============================================

bars_rmse = ax2.bar(
    model_names,
    results['Test RMSE'],
    color=colors
)

ax2.set_title('RMSE Comparison')
ax2.set_ylabel('RMSE')

for bar in bars_rmse:

    ax2.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.003,
        f'{bar.get_height():.4f}',
        ha='center',
        fontsize=8
    )

# ============================================
# 3. MAE
# ============================================

x = np.arange(len(model_names))
width = 0.35

mae_test = ax3.bar(
    x - width/2,
    results['Test MAE'],
    width,
    label='Test MAE'
)

mae_cv = ax3.bar(
    x + width/2,
    results['CV MAE'],
    width,
    label='CV MAE'
)

ax3.set_xticks(x)
ax3.set_xticklabels(model_names)

ax3.set_title('MAE Comparison')
ax3.set_ylabel('MAE')

ax3.legend()

for bar in list(mae_test) + list(mae_cv):

    ax3.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.003,
        f'{bar.get_height():.4f}',
        ha='center',
        fontsize=8
    )

# ============================================
# Final
# ============================================

plt.suptitle(
    'Model Performance Comparison',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()

plt.show()

## 3. Nhận xét và Lựa chọn Champion Model

### 3.1. Phân tích kết quả

Kết quả thực nghiệm cho thấy **XGBoost** đạt hiệu suất tốt nhất với các chỉ số vượt trội:
* **R²** = `0.6082`
* **RMSE** = `0.2761`
* **MAE** = `0.1984`
* **Generalization Gap** ≈ `-0.0063`: Khoảng cách cực nhỏ chứng tỏ mô hình có tính ổn định cao và không bị **Overfitting** nghiêm trọng.

### 3.2. So sánh giữa các mô hình

* **XGBoost vs Random Forest**: Cả hai đều cho kết quả cạnh tranh, nhưng **XGBoost** dẫn đầu trên cả 3 thước đo.
* **Ridge Regression**: Có hiệu suất thấp hơn đáng kể. Điều này khẳng định mối quan hệ giữa các đặc trưng và giá nhà mang tính **phi tuyến** mạnh, vượt quá khả năng biểu diễn của các mô hình tuyến tính.

### 3.3. Giới hạn và Hướng phát triển

Mô hình giải thích được 60.82% biến động, phần còn lại nằm ở các yếu tố ngoại cảnh khó số hóa như:
* **Yếu tố định tính**: Tâm lý thị trường, tính pháp lý, quy hoạch tương lai.
* **Yếu tố thời gian**: Biến động kinh tế theo chu kỳ.
* **Dữ liệu không gian**: Khoảng cách cụ thể tới các tiện ích công cộng (bệnh viện, trường học).

# PHẦN 23: PHÂN TÍCH ĐỘ QUAN TRỌNG CỦA ĐẶC TRƯNG (FEATURE IMPORTANCE)

## 1. Phương pháp

Sử dụng chỉ số `feature_importances_` từ mô hình **XGBoost** để trả lời câu hỏi: **"Yếu tố nào ảnh hưởng mạnh nhất đến giá nhà?"**.

* Chỉ số này dựa trên mức độ giảm **Impurity** trung bình khi đặc trưng đó được dùng để phân chia dữ liệu.
* Giúp xác định các biến đóng góp nhiều nhất vào khả năng dự báo của mô hình.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# ===== 1. Trích xuất và làm sạch tên các thuộc tính =====
feature_names = [
    name.replace('num__', '').replace('remainder__', '')
    for name in preprocessor.get_feature_names_out()
]

# ===== 2. Khởi tạo DataFrame độ quan trọng (Feature Importance) =====
fi_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': best_xgb_model.named_steps['model'].feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

# In bảng xếp hạng thuộc tính ra màn hình console
print(fi_df.to_string(index=False))

# ===== 3. Chuẩn bị dữ liệu trực quan hóa =====
median_val = fi_df['Importance'].median()

# Tạo danh sách màu dựa trên giá trị Median
colors = ['orange' if val >= median_val else 'lightblue' for val in fi_df['Importance']]

# ===== 4. Tiến hành vẽ biểu đồ thanh ngang =====
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(fi_df['Feature'], fi_df['Importance'], color=colors)

# Điền nhãn giá trị tương ứng vào cuối mỗi thanh
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.005, bar.get_y() + bar.get_height() / 2, f'{w:.3f}', va='center', fontsize=9)

# Vẽ đường giới hạn Median phân loại các thuộc tính
ax.axvline(median_val, color='gray', linestyle='--', linewidth=1, label=f'Median: {median_val:.3f}')

ax.set_xlabel('Feature Importance')
ax.set_title('XGBoost Feature Importance - Ảnh hưởng đến Target', fontsize=12, fontweight='bold')
ax.set_xlim(0, fi_df['Importance'].max() + 0.05)
ax.legend()

plt.tight_layout()
plt.show()


## 2. Nhận xét chi tiết Feature Importance

Kết quả cho thấy cấu trúc giá bất động sản tại Hà Nội và TP.HCM phụ thuộc vào các nhóm sau:

### A. Nhóm yếu tố quyết định (Nằm trên đường Median)

* **`district` (0.2357) — Quan trọng nhất**: Vị trí quận/huyện là yếu tố tiên quyết. Nhà nội thành có giá trị cao hơn nhiều so với ngoại thành do ưu thế về hạ tầng.
* **`area` (0.1643) — Quan trọng thứ hai**: Diện tích là biến số cơ bản nhất để định lượng giá trị bất động sản.
* **`street_in_front_of_house` (0.1178)**: Độ rộng mặt đường phản ánh tiềm năng kinh doanh và tính thanh khoản.
* **`floor_number`, `type`, `bedroom_number`**: Phản ánh quy mô và công năng sử dụng thực tế.

### B. Nhóm yếu tố hỗ trợ (Nằm dưới đường Median)

* **`city`, `width`**: Thành phố và chiều rộng mặt tiền đóng vai trò bổ trợ nhưng mức độ ưu tiên thấp hơn các yếu tố trên.
* **Tiện ích nội thất (`is_car_pack`, `is_kitchen`, etc.)**: Có đóng góp thấp nhất. Trong phân khúc dưới 15 tỷ, người mua chú trọng vào **Vị trí** và **Diện tích** trước khi xem xét đến nội thất.

> **Kết luận**: Giá nhà tuân theo quy tắc: **Vị trí (`district`) > Diện tích (`area`) > Tiếp cận đường (`street_in_front_of_house`)**.

# PHẦN 24: TRỰC QUAN HÓA DỰ ĐOÁN VS THỰC TẾ

## 1. Phân tích đồ thị Scatter

Biểu đồ so sánh giá trị dự đoán từ `best_xgb_model` với giá trị thực tế trên tập `Test Set`.

* **Đường đỏ đứt đoạn**: Đại diện cho dự đoán hoàn hảo (Sai số bằng 0).
* **Phân bổ điểm**: Các điểm tập trung sát đường chéo ở vùng giá trung bình cho thấy mô hình dự báo rất tốt ở phân khúc phổ biến.
* **Sai lệch**: Các điểm lệch xa thường nằm ở hai cực (nhà rất rẻ hoặc rất đắt), nơi dữ liệu huấn luyện thưa thớt hơn.

In [ ]:
# Inverse transform về đơn vị tỷ đồng
y_test_real = np.expm1(y_test)
xgb_pred_real = np.expm1(xgb_pred)

plt.figure(figsize=(8, 6))

plt.scatter(y_test_real, xgb_pred_real,
            alpha=0.4, c='blue', edgecolors='white', s=20,
            label='Predictions')

line_coords = [y_test_real.min(), y_test_real.max()]
plt.plot(line_coords, line_coords,
         color='red', linestyle='--', linewidth=2,
         label='Perfect Prediction')

plt.xlabel('Actual Price (tỷ đồng)')
plt.ylabel('Predicted Price (tỷ đồng)')
plt.title('Actual vs Predicted Price — XGBoost')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

> **Ghi chú**: Kết quả này khẳng định mô hình đủ tin cậy để áp dụng cho các bài toán định giá nhanh trên thị trường nhà ở dân dụng.

# PHẦN 25: LƯU TRỮ MÔ HÌNH (MODEL PERSISTENCE)

Chúng ta sử dụng thư viện `joblib` để xuất mô hình `best_xgb_model` ra file `.pkl`, phục vụ cho việc triển khai (deployment) hoặc tái sử dụng sau này mà không cần huấn luyện lại.

In [ ]:
import joblib
import pandas as pd
from datetime import datetime

# =====================================================
# SAVE FULL MODEL PACKAGE
# =====================================================

model_package = {
    # Best model
    'model': best_xgb_model,

    # Model name
    'model_name': 'XGBoost Regressor',

    # Metrics
    'metrics': {
        'r2_score': xgb_r2,
        'rmse': xgb_rmse,
        'mae': xgb_mae,
        'cv_mae': xgb_cv_mae
    },

    # Best hyperparameters
    'best_params': xgb_search.best_params_,

    # Feature names (Sử dụng feature_names đã xử lý ở phần trước)
    'feature_names': feature_names,

    # Dataset information
    'dataset_shape': (len(x_train), len(feature_names)),

    # Additional metadata
    'target': 'price',
    'log_transform': True,
    'created_at': str(datetime.now())
}

# =====================================================
# SAVE TO PKL FILE
# =====================================================

joblib.dump(model_package, 'house_price_xgb_package.pkl')

print("Saved successfully!")
print("File name: house_price_xgb_package.pkl")
display(model_package['metrics'])

In [ ]:
# import joblib

# joblib.dump(best_xgb_model, 'best_xgb_model.pkl')
# print("Model saved: best_xgb_model.pkl")
